In [5]:
!pip install awscli
!pip install h5py

# change both 01 to 02, 03

In [4]:
!aws s3 sync --no-sign-request \
s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03 \
/content/sub-03_ICA


download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_stimulus-metadata.json to sub-03_ICA/voxel-metadata/sub-03_task-things_stimulus-metadata.json
download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_voxel-metadata.json to sub-03_ICA/voxel-metadata/sub-03_task-things_voxel-metadata.json
download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_stimulus-metadata.tsv to sub-03_ICA/voxel-metadata/sub-03_task-things_stimulus-metadata.tsv
download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_voxel-wise-responses.json to sub-03_ICA/voxel-metadata/sub-03_task-things_voxel-wise-responses.json
download: s3://openneuro.org/ds004192/derivatives/ICA-betas/sub-03/voxel-metadata/sub-03_task-things_voxel-metadata.tsv to sub-03_ICA/voxel-metadata/sub-03_task-things_voxel-metadata.tsv
download: s3://openneuro.org/ds0041

In [3]:
!ls /content/sub-01_ICA

voxel-metadata


In [4]:
!ls /content/sub-01_ICA -R | head -n 200

/content/sub-01_ICA:
voxel-metadata

/content/sub-01_ICA/voxel-metadata:
sub-01_task-things_stimulus-metadata.json
sub-01_task-things_stimulus-metadata.tsv
sub-01_task-things_voxel-metadata.json
sub-01_task-things_voxel-metadata.tsv
sub-01_task-things_voxel-wise-responses.h5
sub-01_task-things_voxel-wise-responses.json


In [6]:
import h5py

file_path = "/content/sub-01_ICA/voxel-metadata/sub-01_task-things_voxel-wise-responses.h5"

with h5py.File(file_path, "r") as f:
    print("Keys:", list(f.keys()))

Keys: ['ResponseData']


In [9]:
import h5py

file_path = "/content/sub-01_ICA/voxel-metadata/sub-01_task-things_voxel-wise-responses.h5"

with h5py.File(file_path, "r") as f:
    print("Top level keys:", list(f.keys()))
    print("Inside ResponseData:", list(f["ResponseData"].keys()))

Top level keys: ['ResponseData']
Inside ResponseData: ['axis0', 'axis1', 'block0_items', 'block0_values', 'block1_items', 'block1_values']


In [1]:
import h5py
import numpy as np

file_path = "/content/sub-01_ICA/voxel-metadata/sub-01_task-things_voxel-wise-responses.h5"

with h5py.File(file_path, "r") as f:
    group = f["ResponseData"]
    # Main response matrix
    Y = group["block0_values"][()]
    print("Response matrix shape:", Y.shape)
    print("Data type:", Y.dtype)

Response matrix shape: (211339, 9840)
Data type: float32


In [1]:
import h5py
import pandas as pd
import numpy as np

# HDF5 file
h5_path = "/content/sub-01_ICA/voxel-metadata/sub-01_task-things_voxel-wise-responses.h5"

# Load metadata
stim_meta_path = "/content/sub-01_ICA/voxel-metadata/sub-01_task-things_stimulus-metadata.tsv"
stim_meta = pd.read_csv(stim_meta_path, sep="\t")

with h5py.File(h5_path, "r") as f:
    group = f["ResponseData"]
    Y_all_trials = group["block0_values"][()]  # main matrix

In [2]:
stim_meta.head()

,"trial_type,session,run,subject_id,trial_id,stimulus,concept"
0,"train,1,1,1,0,dog_12s.jpg,dog"
1,"train,1,1,1,1,mango_12s.jpg,mango"
2,"train,1,1,1,2,spatula_12s.jpg,spatula"
3,"test,1,1,1,3,candelabra_14s.jpg,candelabra"
4,"train,1,1,1,4,panda_12s.jpg,panda"


In [6]:
import h5py
import numpy as np
import pandas as pd

# user parameters
sub_id = 3  # change this to 1, 2, or 3
data_root = f"/content/sub-0{sub_id}_ICA"  # set to point to the correct folder

# load data
stim_meta_path = f"{data_root}/voxel-metadata/sub-0{sub_id}_task-things_stimulus-metadata.tsv"
stim_meta = pd.read_csv(stim_meta_path, sep=",")

# load HDF5 respons ematrix
h5_path = f"{data_root}/voxel-metadata/sub-0{sub_id}_task-things_voxel-wise-responses.h5"
with h5py.File(h5_path, "r") as f:
    Y_all_trials = f["ResponseData"]["block0_values"][()]

# average trials per concept
concepts = stim_meta['concept'].unique()
n_voxels = Y_all_trials.shape[1]
Y_objects = np.zeros((len(concepts), n_voxels), dtype=np.float32)
concept_mapping = []

for i, concept in enumerate(concepts):
    idx = stim_meta[stim_meta['concept'] == concept].index
    Y_objects[i] = Y_all_trials[idx].mean(axis=0)
    concept_mapping.append({
        "concept": concept,
        "stimulus_ids": list(stim_meta.loc[idx, 'stimulus']),
        "n_trials": len(idx)
    })

# save
np.save(f"{data_root}/Y_sub0{sub_id}_visual.npy", Y_objects)
pd.DataFrame(concept_mapping).to_csv(f"{data_root}/concepts_sub0{sub_id}.csv", index=False)

print(f"Sub-0{sub_id} done: {Y_objects.shape[0]} concepts x {Y_objects.shape[1]} voxels")

# clear memory for RAM
del Y_all_trials, Y_objects, stim_meta

Sub-03 done: 720 concepts x 9840 voxels


In [7]:
import os

# Path to where you saved your processed .npy and .csv files
root_paths = [
    "/content/sub-01_ICA",
    "/content/sub-02_ICA",
    "/content/sub-03_ICA"
]

for path in root_paths:
    npy_files = [f for f in os.listdir(path) if f.endswith(".npy")]
    csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
    print(f"\n{path}:")
    print("  .npy files:", npy_files)
    print("  .csv files:", csv_files)


/content/sub-01_ICA:
  .npy files: ['Y_sub01_visual.npy']
  .csv files: ['concepts_sub01.csv']

/content/sub-02_ICA:
  .npy files: ['Y_sub02_visual.npy']
  .csv files: ['concepts_sub02.csv']

/content/sub-03_ICA:
  .npy files: ['Y_sub03_visual.npy']
  .csv files: ['concepts_sub03.csv']
